In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.manifold import TSNE
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# SEEDING 
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)       
    np.random.seed(seed)          
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)   

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


# ─────────────────────────────────────────────
# ══ ONLY CHANGE THESE TWO LINES ══
# ─────────────────────────────────────────────
CLEAN_CHECKPOINT = "hybrid_resnet_NOGAN.pth"         
QNI_CHECKPOINT   = "hybrid_resnet_v3_qni_only.pth"   

CLEAN_LABEL = "Clean Baseline"
QNI_LABEL   = "QNI-CCP Trained"


# ─────────────────────────────────────────────
# CONFIG 
# ─────────────────────────────────────────────
n_qubits    = 8
q_depth     = 6
q_out_dim   = 3 * n_qubits   
num_classes = 10
batch_size  = 32              
EPSILON_Q   = 1.0             


# ─────────────────────────────────────────────
# TRANSFORMS 
# ─────────────────────────────────────────────
eval_transform = transforms.Compose([
    transforms.Grayscale(1),              
    transforms.Resize((32, 32)),          
    transforms.ToTensor(),                
    transforms.Normalize((0.5,), (0.5,)) 
])


# ─────────────────────────────────────────────
# DATASETS 
# ─────────────────────────────────────────────
TRAIN_PATH = 'virus_MNIST dataset/train_balanced_v2'
TEST_PATH  = 'virus_MNIST dataset/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    class_names   = test_dataset.classes   
    print(f"Test  samples : {len(test_dataset)}")
    print(f"Train samples : {len(train_dataset)}  (used only for centroids)")
    print(f"Classes ({num_classes}): {class_names}")
except Exception as e:
    print(f"Dataset error: {e}"); raise

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                          shuffle=False, num_workers=4, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# MODEL DEFINITION 
# ══════════════════════════════════════════════════════════════════════════════

dev_qml = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_qml, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[..., i],             wires=i)
        qml.RZ(inputs[..., i + n_qubits], wires=i)
    for l in range(weights.shape[0]):
        if l % 2 == 0:
            for i in range(0, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])
        else:
            for i in range(1, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])
        for i in range(n_qubits):
            qml.RY(weights[l, i, 0] + inputs[..., i],             wires=i)
            qml.RZ(weights[l, i, 1] + inputs[..., i + n_qubits], wires=i)
    measurements = []
    for i in range(n_qubits):
        measurements.append(qml.expval(qml.PauliZ(i)))
        measurements.append(qml.expval(qml.PauliX(i)))
        measurements.append(qml.expval(qml.PauliY(i)))
    return measurements

weight_shapes = {"weights": (q_depth, n_qubits, 3)}

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4), bias=False),
            nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.20, drop_path=0.10):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se             = SEBlock(out_ch)
        self.drop_path_rate = drop_path
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.se(self.conv_block(x))
        if self.training and self.drop_path_rate > 0:
            keep = 1 - self.drop_path_rate
            mask = (torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep).float()
            out  = out * mask / keep
        return self.relu(out + self.skip(x))

class QuantumBridge(nn.Module):
    def __init__(self, in_features, n_qubits):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(in_features, 32), nn.LayerNorm(32),
            nn.GELU(), nn.Dropout(0.35), nn.Linear(32, n_qubits * 2)
        )
        self.angle_scale = nn.Parameter(torch.ones(n_qubits * 2) * torch.pi)
        self.angle_bias  = nn.Parameter(torch.zeros(n_qubits * 2))

    def forward(self, x):
        return self.angle_scale * torch.sigmoid(self.project(x)) + self.angle_bias

class HybridResNet(nn.Module):
    def __init__(self, n_qubits, q_out_dim, num_classes, dropout=0.35):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True)
        )
        self.stage1 = nn.Sequential(ResBlock(16,16,drop_path=0.10), ResBlock(16,16,drop_path=0.10))
        self.stage2 = nn.Sequential(ResBlock(16,32,stride=2,drop_path=0.15), ResBlock(32,32,drop_path=0.15))
        self.stage3 = nn.Sequential(ResBlock(32,64,stride=2,drop_path=0.20), ResBlock(64,64,drop_path=0.20))
        self.gap        = nn.AdaptiveAvgPool2d(1)   
        self.bridge     = QuantumBridge(64, n_qubits)
        self.q_layer    = TorchLayer(quantum_circuit, weight_shapes)
        self.classifier = nn.Sequential(
            nn.Linear(q_out_dim, q_out_dim*2), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(q_out_dim*2, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x); x = self.stage1(x)
        x = self.stage2(x); x = self.stage3(x)
        x = self.gap(x)                   
        x = x.view(x.size(0), -1)         
        x = self.bridge(x)                
        x = self.q_layer(x)               
        return self.classifier(x)         


# ══════════════════════════════════════════════════════════════════════════════
# SHARED UTILITIES 
# ══════════════════════════════════════════════════════════════════════════════

class FeatureHook:
    def __init__(self, layer):
        self.features = None
        self._handle  = layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'features', o)
        )
    def remove(self): self._handle.remove()

def extract_all(model, dataloader, device):
    model.eval()
    all_features, all_labels, all_preds, all_probs = [], [], [], []
    hook = FeatureHook(model.gap) 

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="  Extracting", leave=False):
            x, y   = x.to(device), y.to(device)
            logits = model(x)                                    
            z      = hook.features.view(hook.features.size(0), -1) 
            probs  = F.softmax(logits, dim=1)                    

            all_features.append(z.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            all_preds.append(logits.argmax(1).cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    hook.remove()
    return (
        np.concatenate(all_features, 0),   
        np.concatenate(all_labels,   0),   
        np.concatenate(all_preds,    0),   
        np.concatenate(all_probs,    0)    
    )

def compute_class_centroids(model, dataloader, device, num_classes):
    model.eval()
    sum_f = torch.zeros(num_classes, 64, device=device)
    count = torch.zeros(num_classes,     device=device)
    hook  = FeatureHook(model.gap)

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="  Centroids", leave=False):
            x, y = x.to(device), y.to(device)
            _    = model(x)
            z    = hook.features.view(hook.features.size(0), -1)
            for c in range(num_classes):
                mask = (y == c)
                if mask.sum() > 0:
                    sum_f[c] += z[mask].sum(0)
                    count[c] += mask.sum()

    hook.remove()
    return sum_f / count.clamp(min=1.0).unsqueeze(1)

def evaluate_qni_robustness(model, dataloader, device, centroids, epsilon_q):
    model.eval()
    correct, total = 0, 0

    for x, y in tqdm(dataloader, desc="  QNI Robustness", leave=False):
        x, y  = x.to(device), y.to(device)
        hook  = FeatureHook(model.gap)
        _     = model(x)
        z_raw = hook.features.view(hook.features.size(0), -1)
        hook.remove()

        z = z_raw.detach().requires_grad_(True)
        logits = model.classifier(model.q_layer(model.bridge(z)))
        F.cross_entropy(logits, y).backward()

        S_norm = z.grad.detach()
        S_norm = S_norm / (S_norm.norm(dim=1, keepdim=True) + 1e-8)

        z_det  = z.detach()
        wrong  = [np.random.choice(
                    [c for c in range(centroids.size(0)) if c != yi.item()])
                  for yi in y]
        mu_w   = centroids[torch.tensor(wrong, device=device)]
        z_pert = z_det + epsilon_q * (S_norm * (mu_w - z_det))

        with torch.no_grad():
            preds   = model.classifier(model.q_layer(model.bridge(z_pert))).argmax(1)
            correct += (preds == y).sum().item()
            total   += y.size(0)

    return correct / total


# ══════════════════════════════════════════════════════════════════════════════
# REWRITTEN TO MATCH CODE 2 METRICS LOGIC
# ══════════════════════════════════════════════════════════════════════════════
def compute_latent_metrics(features_np, probs_np):
    """
    Computes latent metrics using the exact PyTorch logic from Code 2.
    """
    # Convert incoming NumPy arrays back to PyTorch tensors to match Code 2
    features = torch.tensor(features_np)
    probs = torch.tensor(probs_np)

    # 1. Entropy (Code 2 style)
    # Uses torch.sum and torch.log with 1e-8 epsilon
    entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=1).mean().item()

    # 2. Feature Variance (Code 2 style)
    # Uses PyTorch's torch.var (which calculates Sample Variance, not Population Variance)
    feature_variance = torch.var(features, dim=0).mean().item()

    # 3. Unique Patterns (Code 2 style)
    # Counts the number of unique predicted classes, not unique feature templates
    unique_patterns = len(torch.unique(probs.argmax(1)))

    # 4. Confidence Metrics (Code 2 style)
    max_probs = probs.max(1)[0]
    confidences = max_probs.cpu().tolist()

    # High confidence (Code 2 style - median split)
    threshold = np.percentile(confidences, 50)
    high_conf_pct = np.mean([c > threshold for c in confidences]) * 100

    avg_confidence = np.mean(confidences)
    conf_std = np.std(confidences)

    return {
        'entropy'          : entropy,
        'feature_variance' : feature_variance,
        'unique_patterns'  : unique_patterns,
        'high_conf_pct'    : high_conf_pct,
        'avg_confidence'   : avg_confidence,
        'conf_std'         : conf_std,
    }


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — FULL MODEL EVALUATION 
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_model(model, label, test_loader, train_loader, device):
    print(f"\n{'─'*60}")
    print(f"  Evaluating: {label}")
    print(f"{'─'*60}")

    features, true_labels, pred_labels, pred_probs = \
        extract_all(model, test_loader, device)
    print(f"  Features extracted: {features.shape}")

    overall_acc = accuracy_score(true_labels, pred_labels)
    report_str  = classification_report(
        true_labels, pred_labels,
        target_names=class_names, digits=4
    )
    report_dict = classification_report(
        true_labels, pred_labels,
        target_names=class_names, digits=4, output_dict=True
    )

    print(f"\n  Classification Report — {label}")
    print(report_str)

    print(f"  Computing centroids for QNI robustness...")
    centroids   = compute_class_centroids(model, train_loader, device, num_classes)
    qni_acc     = evaluate_qni_robustness(model, test_loader, device, centroids, EPSILON_Q)
    robust_drop = (overall_acc - qni_acc) * 100   

    print(f"  Clean Accuracy       : {overall_acc:.4f}")
    print(f"  QNI Robustness Acc   : {qni_acc:.4f}")
    print(f"  Robustness drop      : {robust_drop:.2f} pp")

    lm = compute_latent_metrics(features, pred_probs)

    print(f"\n  Running t-SNE...")
    seed_all(42)   
    tsne = TSNE(
        n_components = 2,    
        perplexity   = 40,   
        n_iter       = 1000, 
        random_state = 42,   
        learning_rate= 200,  
        init         = 'pca' 
    )
    tsne_emb = tsne.fit_transform(features)   
    print(f"  t-SNE complete: {tsne_emb.shape}")

    return {
        'label'       : label,
        'features'    : features,       
        'tsne_emb'    : tsne_emb,       
        'true_labels' : true_labels,    
        'pred_labels' : pred_labels,    
        'pred_probs'  : pred_probs,     
        'clean_acc'   : overall_acc,
        'qni_acc'     : qni_acc,
        'robust_drop' : robust_drop,
        'report_dict' : report_dict,
        'latent'      : lm,
    }


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — INDIVIDUAL PLOTS 
# ══════════════════════════════════════════════════════════════════════════════

PALETTE = plt.cm.get_cmap('tab10', num_classes)   

def _style_ax(ax):
    ax.set_facecolor('#0f0f1a')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333355')
    ax.tick_params(colors='#888899')
    ax.yaxis.grid(True, linestyle=':', alpha=0.25, color='#555577')
    ax.set_axisbelow(True)

def plot_tsne_single(result, save_path):
    emb    = result['tsne_emb']       
    labels = result['true_labels']    
    label  = result['label']

    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#0f0f1a')

    for c in range(num_classes):
        mask = (labels == c)
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=[PALETTE(c)], label=class_names[c],
                   s=16, alpha=0.82, edgecolors='none')

    ax.set_title(f"t-SNE of GAP Features\n{label}",
                 fontsize=13, fontweight='bold', color='white', pad=12)
    ax.set_xlabel("t-SNE Dim 1", fontsize=10, color='#aaaacc')
    ax.set_ylabel("t-SNE Dim 2", fontsize=10, color='#aaaacc')
    _style_ax(ax)
    leg = ax.legend(title="Malware Family", fontsize=8, title_fontsize=9,
                    facecolor='#1a1a2e', edgecolor='#444466',
                    framealpha=0.4, labelcolor='white', loc='upper right')
    leg.get_title().set_color('white')
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_confusion_single(result, save_path):
    cm      = confusion_matrix(result['true_labels'], result['pred_labels'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) 

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#0f0f1a')

    sns.heatmap(cm_norm, annot=True, fmt='.2f',
                cmap='YlOrRd',                    
                vmin=0.0, vmax=1.0,               
                xticklabels=class_names,
                yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor='#1a1a2e',
                cbar_kws={'shrink': 0.8})

    ax.set_title(f"Confusion Matrix (row-normalised)\n{result['label']}",
                 fontsize=12, fontweight='bold', color='white', pad=12)
    ax.set_xlabel("Predicted", fontsize=10, color='#aaaacc', labelpad=8)
    ax.set_ylabel("True Label", fontsize=10, color='#aaaacc', labelpad=8)
    ax.tick_params(axis='x', colors='#ccccee', labelsize=8, rotation=40)
    ax.tick_params(axis='y', colors='#ccccee', labelsize=8, rotation=0)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=8, colors='#aaaacc')
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_f1_single(result, save_path):
    rd        = result['report_dict']
    f1_scores = [rd[cls]['f1-score'] for cls in class_names]
    macro_f1  = rd['macro avg']['f1-score']

    fig, ax = plt.subplots(figsize=(11, 4))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#0f0f1a')

    bar_colors = ['#4ade80' if f >= macro_f1 else '#fb923c' for f in f1_scores]
    bars = ax.bar(range(num_classes), f1_scores, color=bar_colors,
                  edgecolor='#222244', width=0.65)
    for bar, val in zip(bars, f1_scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom',
                fontsize=8, color='white', fontweight='bold')

    ax.axhline(y=macro_f1, color='#60a5fa', linestyle='--', linewidth=1.5,
               label=f'Macro F1 = {macro_f1:.3f}')
    ax.set_xticks(range(num_classes))
    ax.set_xticklabels(class_names, rotation=35, ha='right',
                       fontsize=8, color='#ccccee')
    ax.set_ylim(0, 1.08)   
    ax.set_ylabel("F1-Score", fontsize=10, color='#aaaacc')
    ax.set_title(f"Per-Class F1-Score\n{result['label']}",
                 fontsize=12, fontweight='bold', color='white', pad=10)
    ax.legend(fontsize=9, labelcolor='white', facecolor='#1a1a2e',
              edgecolor='#444466', framealpha=0.4)
    ax.tick_params(axis='y', colors='#aaaacc')
    _style_ax(ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — SIDE-BY-SIDE COMPARISON PLOTS
# ══════════════════════════════════════════════════════════════════════════════

def plot_tsne_comparison(r_clean, r_qni, save_path="comparison_tsne.png"):
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle(
        "t-SNE of GAP Features: Clean Baseline vs QNI-CCP Trained\n"
        "(Same random_state=42 — layout differences reflect learned feature structure)",
        fontsize=13, fontweight='bold', color='white', y=1.01
    )

    for ax, result in zip(axes, [r_clean, r_qni]):
        ax.set_facecolor('#0f0f1a')
        emb    = result['tsne_emb']
        labels = result['true_labels']

        for c in range(num_classes):
            mask = (labels == c)
            ax.scatter(emb[mask, 0], emb[mask, 1],
                       c=[PALETTE(c)], label=class_names[c],
                       s=16, alpha=0.82, edgecolors='none')

        stats = (f"Clean Acc: {result['clean_acc']:.4f}\n"
                 f"QNI Rob:   {result['qni_acc']:.4f}\n"
                 f"Δ drop:    {result['robust_drop']:.2f} pp")
        ax.text(0.02, 0.98, stats, transform=ax.transAxes,
                fontsize=9, color='#ccddff', va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#1a1a2e',
                          edgecolor='#444466', alpha=0.7))

        ax.set_title(result['label'], fontsize=12, fontweight='bold',
                     color='white', pad=10)
        ax.set_xlabel("t-SNE Dim 1", fontsize=10, color='#aaaacc')
        ax.set_ylabel("t-SNE Dim 2", fontsize=10, color='#aaaacc')
        _style_ax(ax)

    handles = [plt.scatter([], [], c=[PALETTE(c)], s=30, label=class_names[c])
               for c in range(num_classes)]
    fig.legend(handles=handles, title="Malware Family", fontsize=9,
               title_fontsize=10, ncol=5, loc='lower center',
               bbox_to_anchor=(0.5, -0.06),
               facecolor='#1a1a2e', edgecolor='#444466',
               framealpha=0.5, labelcolor='white').get_title().set_color('white')

    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_confusion_comparison(r_clean, r_qni,
                               save_path="comparison_confusion_matrix.png"):
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle("Confusion Matrix Comparison (row-normalised)",
                 fontsize=13, fontweight='bold', color='white')

    for ax, result in zip(axes, [r_clean, r_qni]):
        cm      = confusion_matrix(result['true_labels'], result['pred_labels'])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        ax.set_facecolor('#0f0f1a')

        sns.heatmap(cm_norm, annot=True, fmt='.2f',
                    cmap='YlOrRd', vmin=0.0, vmax=1.0,   
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.4, linecolor='#1a1a2e',
                    cbar_kws={'shrink': 0.8})

        ax.set_title(result['label'], fontsize=12, fontweight='bold',
                     color='white', pad=10)
        ax.set_xlabel("Predicted", fontsize=10, color='#aaaacc', labelpad=6)
        ax.set_ylabel("True Label", fontsize=10, color='#aaaacc', labelpad=6)
        ax.tick_params(axis='x', colors='#ccccee', labelsize=8, rotation=40)
        ax.tick_params(axis='y', colors='#ccccee', labelsize=8, rotation=0)
        ax.collections[0].colorbar.ax.tick_params(labelsize=8, colors='#aaaacc')

    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_f1_comparison(r_clean, r_qni,
                        save_path="comparison_per_class_f1.png"):
    rd_c = r_clean['report_dict']
    rd_q = r_qni['report_dict']

    f1_clean = np.array([rd_c[cls]['f1-score'] for cls in class_names])
    f1_qni   = np.array([rd_q[cls]['f1-score'] for cls in class_names])
    x        = np.arange(num_classes)
    w        = 0.38   

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#0f0f1a')

    bars_c = ax.bar(x - w/2, f1_clean, w, label=CLEAN_LABEL,
                    color='#f87171', edgecolor='#222244', alpha=0.9)
    bars_q = ax.bar(x + w/2, f1_qni,   w, label=QNI_LABEL,
                    color='#4ade80', edgecolor='#222244', alpha=0.9)

    for bar in bars_c:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom',
                fontsize=7, color='#ffaaaa')
    for bar in bars_q:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom',
                fontsize=7, color='#aaffaa')

    ax.axhline(rd_c['macro avg']['f1-score'], color='#f87171',
               linestyle='--', linewidth=1.2, alpha=0.6,
               label=f'Clean macro avg = {rd_c["macro avg"]["f1-score"]:.3f}')
    ax.axhline(rd_q['macro avg']['f1-score'], color='#4ade80',
               linestyle='--', linewidth=1.2, alpha=0.6,
               label=f'QNI macro avg = {rd_q["macro avg"]["f1-score"]:.3f}')

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=35, ha='right',
                       fontsize=8, color='#ccccee')
    ax.set_ylim(0, 1.10)
    ax.set_ylabel("F1-Score", fontsize=10, color='#aaaacc')
    ax.set_title("Per-Class F1-Score: Clean Baseline vs QNI-CCP Trained",
                 fontsize=12, fontweight='bold', color='white', pad=10)
    ax.legend(fontsize=9, labelcolor='white', facecolor='#1a1a2e',
              edgecolor='#444466', framealpha=0.5, ncol=2)
    ax.tick_params(axis='y', colors='#aaaacc')
    _style_ax(ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_latent_metrics_comparison(r_clean, r_qni,
                                    save_path="comparison_latent_metrics.png"):
    lm_c = r_clean['latent']
    lm_q = r_qni['latent']

    metrics_scalar = [
        ('entropy',          'Entropy',               False), 
        ('feature_variance', 'Feature Variance',        False), 
        ('avg_confidence',   'Avg Confidence',          True),  
        ('conf_std',         'Confidence Std Dev',      False), 
    ]
    metrics_pct = [
        ('high_conf_pct',    'High-Conf Patterns (%)', True),  
    ]

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle(
        "Latent-Space Metrics Comparison (Table 6 style)\n"
        f"{CLEAN_LABEL}  vs  {QNI_LABEL}",
        fontsize=12, fontweight='bold', color='white'
    )

    all_metrics = metrics_scalar + metrics_pct
    for i, (key, title, higher_is_better) in enumerate(all_metrics):
        ax  = axes[i]
        ax.set_facecolor('#0f0f1a')

        val_c = lm_c[key]
        val_q = lm_q[key]

        if higher_is_better:
            col_c = '#4ade80' if val_c >= val_q else '#f87171'
            col_q = '#4ade80' if val_q >= val_c else '#f87171'
        else:
            col_c = '#4ade80' if val_c <= val_q else '#f87171'
            col_q = '#4ade80' if val_q <= val_c else '#f87171'

        bars = ax.bar([CLEAN_LABEL, QNI_LABEL], [val_c, val_q],
                      color=[col_c, col_q], edgecolor='#222244', width=0.5)
        for bar, val in zip(bars, [val_c, val_q]):
            label = f'{val:.1f}%' if key == 'high_conf_pct' else f'{val:.4f}'
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(val_c, val_q) * 0.03,
                    label, ha='center', va='bottom',
                    fontsize=9, color='white', fontweight='bold')

        ax.set_title(title, fontsize=10, fontweight='bold',
                     color='white', pad=8)
        ax.set_ylim(0, max(val_c, val_q) * 1.25 + 1e-6)
        ax.tick_params(axis='x', colors='#ccccee', labelsize=8, rotation=15)
        ax.tick_params(axis='y', colors='#888899', labelsize=8)
        _style_ax(ax)

    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY TABLE 
# ══════════════════════════════════════════════════════════════════════════════

def print_comparison_table(r_clean, r_qni):
    rc = r_clean
    rq = r_qni

    def delta(a, b, higher_better=True):
        d = b - a
        arrow = '↑' if (d > 0) == higher_better else '↓'
        return f"{d:+.4f} {arrow}"

    print("\n" + "═"*72)
    print(f"  COMPARISON SUMMARY: {CLEAN_LABEL}  →  {QNI_LABEL}")
    print("═"*72)
    print(f"  {'Metric':<35} {'Clean':>10} {'QNI-CCP':>10} {'Δ':>14}")
    print(f"  {'─'*65}")

    print(f"  {'Clean Test Accuracy':<35} "
          f"{rc['clean_acc']:>10.4f} {rq['clean_acc']:>10.4f} "
          f"{delta(rc['clean_acc'], rq['clean_acc']):>14}")
    print(f"  {'QNI Robustness Accuracy':<35} "
          f"{rc['qni_acc']:>10.4f} {rq['qni_acc']:>10.4f} "
          f"{delta(rc['qni_acc'], rq['qni_acc']):>14}")
    print(f"  {'Robustness Drop (pp)':<35} "
          f"{rc['robust_drop']:>10.2f} {rq['robust_drop']:>10.2f} "
          f"{delta(rc['robust_drop'], rq['robust_drop'], higher_better=False):>14}")

    print(f"  {'─'*65}")
    for key, name, hb in [
        ('macro avg',    'Macro Precision',  True),
        ('macro avg',    'Macro Recall',     True),
        ('macro avg',    'Macro F1',         True),
        ('weighted avg', 'Weighted F1',      True),
    ]:
        sub = 'precision' if 'Precision' in name else \
              'recall'    if 'Recall'    in name else 'f1-score'
        vc = rc['report_dict'][key][sub]
        vq = rq['report_dict'][key][sub]
        print(f"  {name:<35} {vc:>10.4f} {vq:>10.4f} {delta(vc, vq, hb):>14}")

    print(f"  {'─'*65}")
    for key, name, hb in [
        ('entropy',          'Entropy',              False),
        ('feature_variance', 'Feature Variance',     False),
        ('unique_patterns',  'Unique Predicted Cls', None), # Updated Name
        ('high_conf_pct',    'High-Conf (Median) %', True), # Updated Name
        ('avg_confidence',   'Avg Confidence',       True),
        ('conf_std',         'Confidence Std Dev',   False),
    ]:
        vc = rc['latent'][key]
        vq = rq['latent'][key]
        if key == 'high_conf_pct':
            print(f"  {name:<35} {vc:>9.1f}% {vq:>9.1f}% "
                  f"{delta(vc, vq, hb):>14}")
        elif key == 'unique_patterns':
            print(f"  {name:<35} {int(vc):>10d} {int(vq):>10d}            —")
        else:
            print(f"  {name:<35} {vc:>10.4f} {vq:>10.4f} "
                  f"{delta(vc, vq, hb) if hb is not None else '—':>14}")

    print("═"*72)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def load_model(path, label):
    m = HybridResNet(n_qubits, q_out_dim, num_classes, dropout=0.35).to(device)
    ckpt = torch.load(path, map_location=device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    print(f"  Loaded [{label}] from {path}  "
          f"(epoch={ckpt.get('epoch','?')}, val_acc={ckpt.get('val_acc','?')})")
    return m

print("\n" + "═"*60)
print("  LOADING MODELS")
print("═"*60)
model_clean = load_model(CLEAN_CHECKPOINT, CLEAN_LABEL)
model_qni   = load_model(QNI_CHECKPOINT,   QNI_LABEL)

print("\n" + "═"*60)
print("  EVALUATING CLEAN BASELINE")
print("═"*60)
r_clean = evaluate_model(model_clean, CLEAN_LABEL, test_loader, train_loader, device)

print("\n" + "═"*60)
print("  EVALUATING QNI-CCP MODEL")
print("═"*60)
r_qni = evaluate_model(model_qni, QNI_LABEL, test_loader, train_loader, device)

print_comparison_table(r_clean, r_qni)

print("\n" + "═"*60)
print("  SAVING INDIVIDUAL PLOTS")
print("═"*60)
plot_tsne_single(r_clean, "clean_tsne.png")
plot_tsne_single(r_qni,   "qni_tsne.png")
plot_confusion_single(r_clean, "clean_confusion_matrix.png")
plot_confusion_single(r_qni,   "qni_confusion_matrix.png")
plot_f1_single(r_clean, "clean_per_class_f1.png")
plot_f1_single(r_qni,   "qni_per_class_f1.png")

print("\n" + "═"*60)
print("  SAVING COMPARISON PLOTS")
print("═"*60)
plot_tsne_comparison(r_clean, r_qni,        "comparison_tsne.png")
plot_confusion_comparison(r_clean, r_qni,   "comparison_confusion_matrix.png")
plot_f1_comparison(r_clean, r_qni,          "comparison_per_class_f1.png")
plot_latent_metrics_comparison(r_clean, r_qni, "comparison_latent_metrics.png")

print("\n✅ All evaluations complete.")

Device: cuda
Test  samples : 3458
Train samples : 48498  (used only for centroids)
Classes (10): ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

════════════════════════════════════════════════════════════
  LOADING MODELS
════════════════════════════════════════════════════════════
  Loaded [Clean Baseline] from hybrid_resnet_NOGAN.pth  (epoch=55, val_acc=0.9177175935497209)
  Loaded [QNI-CCP Trained] from hybrid_resnet_v3_qni_only.pth  (epoch=56, val_acc=0.927434360140583)

════════════════════════════════════════════════════════════
  EVALUATING CLEAN BASELINE
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Evaluating: Clean Baseline
────────────────────────────────────────────────────────────


  Features extracted: (3458, 64)

  Classification Report — Clean Baseline
              precision    recall  f1-score   support

           0     0.5000    0.4171    0.4548       175
           1     0.9980    0.9960    0.9970       496
           2     0.8500    0.9951    0.9169       205
           3     0.9497    0.9659    0.9577       176
           4     1.0000    1.0000    1.0000        58
           5     0.9265    0.9123    0.9193       456
           6     0.9649    0.9332    0.9488      1003
           7     0.8987    0.8676    0.8829       491
           8     0.8602    0.9249    0.8914       173
           9     0.8199    0.9511    0.8807       225

    accuracy                         0.9112      3458
   macro avg     0.8768    0.8963    0.8849      3458
weighted avg     0.9100    0.9112    0.9096      3458

  Computing centroids for QNI robustness...


  Clean Accuracy       : 0.9112
  QNI Robustness Acc   : 0.9057
  Robustness drop      : 0.55 pp

  Running t-SNE...
  t-SNE complete: (3458, 2)

════════════════════════════════════════════════════════════
  EVALUATING QNI-CCP MODEL
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Evaluating: QNI-CCP Trained
────────────────────────────────────────────────────────────


  Features extracted: (3458, 64)

  Classification Report — QNI-CCP Trained
              precision    recall  f1-score   support

           0     0.5926    0.0914    0.1584       175
           1     0.9900    0.9960    0.9930       496
           2     0.8772    0.9756    0.9238       205
           3     0.9503    0.9773    0.9636       176
           4     1.0000    1.0000    1.0000        58
           5     0.9300    0.9320    0.9310       456
           6     0.9391    0.9840    0.9611      1003
           7     0.8533    0.9593    0.9032       491
           8     0.8659    0.8960    0.8807       173
           9     0.8938    0.8978    0.8958       225

    accuracy                         0.9196      3458
   macro avg     0.8892    0.8709    0.8610      3458
weighted avg     0.9068    0.9196    0.9031      3458

  Computing centroids for QNI robustness...


  Clean Accuracy       : 0.9196
  QNI Robustness Acc   : 0.9265
  Robustness drop      : -0.69 pp

  Running t-SNE...
  t-SNE complete: (3458, 2)

════════════════════════════════════════════════════════════════════════
  COMPARISON SUMMARY: Clean Baseline  →  QNI-CCP Trained
════════════════════════════════════════════════════════════════════════
  Metric                                   Clean    QNI-CCP              Δ
  ─────────────────────────────────────────────────────────────────
  Clean Test Accuracy                     0.9112     0.9196      +0.0084 ↑
  QNI Robustness Accuracy                 0.9057     0.9265      +0.0208 ↑
  Robustness Drop (pp)                      0.55      -0.69      -1.2435 ↑
  ─────────────────────────────────────────────────────────────────
  Macro Precision                         0.8768     0.8892      +0.0124 ↑
  Macro Recall                            0.8963     0.8709      -0.0254 ↓
  Macro F1                                0.8849     0.8610     

In [ ]:
'''
here QNI applied after baseline training and comapring it with the previous model where the QNI was introduced from the 1st epoch

'''

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.manifold import TSNE
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# SEEDING 
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)       
    np.random.seed(seed)          
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)   

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


# ─────────────────────────────────────────────
# ══ ONLY CHANGE THESE TWO LINES ══
# ─────────────────────────────────────────────
CLEAN_CHECKPOINT = "hybrid_resnet_NOGAN.pth"         
QNI_CHECKPOINT   = "QNI1.pth" 

CLEAN_LABEL = "Clean Baseline"
QNI_LABEL   = "QNI-CCP Trained"


# ─────────────────────────────────────────────
# CONFIG 
# ─────────────────────────────────────────────
n_qubits    = 8
q_depth     = 6
q_out_dim   = 3 * n_qubits   
num_classes = 10
batch_size  = 32              
EPSILON_Q   = 1.0             


# ─────────────────────────────────────────────
# TRANSFORMS 
# ─────────────────────────────────────────────
eval_transform = transforms.Compose([
    transforms.Grayscale(1),              
    transforms.Resize((32, 32)),          
    transforms.ToTensor(),                
    transforms.Normalize((0.5,), (0.5,)) 
])


# ─────────────────────────────────────────────
# DATASETS 
# ─────────────────────────────────────────────
TRAIN_PATH = 'virus_MNIST dataset/train_balanced_v2'
TEST_PATH  = 'virus_MNIST dataset/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    class_names   = test_dataset.classes   
    print(f"Test  samples : {len(test_dataset)}")
    print(f"Train samples : {len(train_dataset)}  (used only for centroids)")
    print(f"Classes ({num_classes}): {class_names}")
except Exception as e:
    print(f"Dataset error: {e}"); raise

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                          shuffle=False, num_workers=4, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# MODEL DEFINITION 
# ══════════════════════════════════════════════════════════════════════════════

dev_qml = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_qml, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[..., i],             wires=i)
        qml.RZ(inputs[..., i + n_qubits], wires=i)
    for l in range(weights.shape[0]):
        if l % 2 == 0:
            for i in range(0, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])
        else:
            for i in range(1, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])
        for i in range(n_qubits):
            qml.RY(weights[l, i, 0] + inputs[..., i],             wires=i)
            qml.RZ(weights[l, i, 1] + inputs[..., i + n_qubits], wires=i)
    measurements = []
    for i in range(n_qubits):
        measurements.append(qml.expval(qml.PauliZ(i)))
        measurements.append(qml.expval(qml.PauliX(i)))
        measurements.append(qml.expval(qml.PauliY(i)))
    return measurements

weight_shapes = {"weights": (q_depth, n_qubits, 3)}

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4), bias=False),
            nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.20, drop_path=0.10):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se             = SEBlock(out_ch)
        self.drop_path_rate = drop_path
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.se(self.conv_block(x))
        if self.training and self.drop_path_rate > 0:
            keep = 1 - self.drop_path_rate
            mask = (torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep).float()
            out  = out * mask / keep
        return self.relu(out + self.skip(x))

class QuantumBridge(nn.Module):
    def __init__(self, in_features, n_qubits):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(in_features, 32), nn.LayerNorm(32),
            nn.GELU(), nn.Dropout(0.35), nn.Linear(32, n_qubits * 2)
        )
        self.angle_scale = nn.Parameter(torch.ones(n_qubits * 2) * torch.pi)
        self.angle_bias  = nn.Parameter(torch.zeros(n_qubits * 2))

    def forward(self, x):
        return self.angle_scale * torch.sigmoid(self.project(x)) + self.angle_bias

class HybridResNet(nn.Module):
    def __init__(self, n_qubits, q_out_dim, num_classes, dropout=0.35):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True)
        )
        self.stage1 = nn.Sequential(ResBlock(16,16,drop_path=0.10), ResBlock(16,16,drop_path=0.10))
        self.stage2 = nn.Sequential(ResBlock(16,32,stride=2,drop_path=0.15), ResBlock(32,32,drop_path=0.15))
        self.stage3 = nn.Sequential(ResBlock(32,64,stride=2,drop_path=0.20), ResBlock(64,64,drop_path=0.20))
        self.gap        = nn.AdaptiveAvgPool2d(1)   
        self.bridge     = QuantumBridge(64, n_qubits)
        self.q_layer    = TorchLayer(quantum_circuit, weight_shapes)
        self.classifier = nn.Sequential(
            nn.Linear(q_out_dim, q_out_dim*2), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(q_out_dim*2, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x); x = self.stage1(x)
        x = self.stage2(x); x = self.stage3(x)
        x = self.gap(x)                   
        x = x.view(x.size(0), -1)         
        x = self.bridge(x)                
        x = self.q_layer(x)               
        return self.classifier(x)         


# ══════════════════════════════════════════════════════════════════════════════
# SHARED UTILITIES 
# ══════════════════════════════════════════════════════════════════════════════

class FeatureHook:
    def __init__(self, layer):
        self.features = None
        self._handle  = layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'features', o)
        )
    def remove(self): self._handle.remove()

def extract_all(model, dataloader, device):
    model.eval()
    all_features, all_labels, all_preds, all_probs = [], [], [], []
    hook = FeatureHook(model.gap) 

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="  Extracting", leave=False):
            x, y   = x.to(device), y.to(device)
            logits = model(x)                                    
            z      = hook.features.view(hook.features.size(0), -1) 
            probs  = F.softmax(logits, dim=1)                    

            all_features.append(z.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            all_preds.append(logits.argmax(1).cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    hook.remove()
    return (
        np.concatenate(all_features, 0),   
        np.concatenate(all_labels,   0),   
        np.concatenate(all_preds,    0),   
        np.concatenate(all_probs,    0)    
    )

def compute_class_centroids(model, dataloader, device, num_classes):
    model.eval()
    sum_f = torch.zeros(num_classes, 64, device=device)
    count = torch.zeros(num_classes,     device=device)
    hook  = FeatureHook(model.gap)

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="  Centroids", leave=False):
            x, y = x.to(device), y.to(device)
            _    = model(x)
            z    = hook.features.view(hook.features.size(0), -1)
            for c in range(num_classes):
                mask = (y == c)
                if mask.sum() > 0:
                    sum_f[c] += z[mask].sum(0)
                    count[c] += mask.sum()

    hook.remove()
    return sum_f / count.clamp(min=1.0).unsqueeze(1)

def evaluate_qni_robustness(model, dataloader, device, centroids, epsilon_q):
    model.eval()
    correct, total = 0, 0

    for x, y in tqdm(dataloader, desc="  QNI Robustness", leave=False):
        x, y  = x.to(device), y.to(device)
        hook  = FeatureHook(model.gap)
        _     = model(x)
        z_raw = hook.features.view(hook.features.size(0), -1)
        hook.remove()

        z = z_raw.detach().requires_grad_(True)
        logits = model.classifier(model.q_layer(model.bridge(z)))
        F.cross_entropy(logits, y).backward()

        S_norm = z.grad.detach()
        S_norm = S_norm / (S_norm.norm(dim=1, keepdim=True) + 1e-8)

        z_det  = z.detach()
        wrong  = [np.random.choice(
                    [c for c in range(centroids.size(0)) if c != yi.item()])
                  for yi in y]
        mu_w   = centroids[torch.tensor(wrong, device=device)]
        z_pert = z_det + epsilon_q * (S_norm * (mu_w - z_det))

        with torch.no_grad():
            preds   = model.classifier(model.q_layer(model.bridge(z_pert))).argmax(1)
            correct += (preds == y).sum().item()
            total   += y.size(0)

    return correct / total


# ══════════════════════════════════════════════════════════════════════════════
# REWRITTEN TO MATCH CODE 2 METRICS LOGIC
# ══════════════════════════════════════════════════════════════════════════════
def compute_latent_metrics(features_np, probs_np):
    """
    Computes latent metrics using the exact PyTorch logic from Code 2.
    """
    # Convert incoming NumPy arrays back to PyTorch tensors to match Code 2
    features = torch.tensor(features_np)
    probs = torch.tensor(probs_np)

    # 1. Entropy (Code 2 style)
    # Uses torch.sum and torch.log with 1e-8 epsilon
    entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=1).mean().item()

    # 2. Feature Variance (Code 2 style)
    # Uses PyTorch's torch.var (which calculates Sample Variance, not Population Variance)
    feature_variance = torch.var(features, dim=0).mean().item()

    # 3. Unique Patterns (Code 2 style)
    # Counts the number of unique predicted classes, not unique feature templates
    unique_patterns = len(torch.unique(probs.argmax(1)))

    # 4. Confidence Metrics (Code 2 style)
    max_probs = probs.max(1)[0]
    confidences = max_probs.cpu().tolist()

    # High confidence (Code 2 style - median split)
    threshold = np.percentile(confidences, 50)
    high_conf_pct = np.mean([c > threshold for c in confidences]) * 100

    avg_confidence = np.mean(confidences)
    conf_std = np.std(confidences)

    return {
        'entropy'          : entropy,
        'feature_variance' : feature_variance,
        'unique_patterns'  : unique_patterns,
        'high_conf_pct'    : high_conf_pct,
        'avg_confidence'   : avg_confidence,
        'conf_std'         : conf_std,
    }


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — FULL MODEL EVALUATION 
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_model(model, label, test_loader, train_loader, device):
    print(f"\n{'─'*60}")
    print(f"  Evaluating: {label}")
    print(f"{'─'*60}")

    features, true_labels, pred_labels, pred_probs = \
        extract_all(model, test_loader, device)
    print(f"  Features extracted: {features.shape}")

    overall_acc = accuracy_score(true_labels, pred_labels)
    report_str  = classification_report(
        true_labels, pred_labels,
        target_names=class_names, digits=4
    )
    report_dict = classification_report(
        true_labels, pred_labels,
        target_names=class_names, digits=4, output_dict=True
    )

    print(f"\n  Classification Report — {label}")
    print(report_str)

    print(f"  Computing centroids for QNI robustness...")
    centroids   = compute_class_centroids(model, train_loader, device, num_classes)
    qni_acc     = evaluate_qni_robustness(model, test_loader, device, centroids, EPSILON_Q)
    robust_drop = (overall_acc - qni_acc) * 100   

    print(f"  Clean Accuracy       : {overall_acc:.4f}")
    print(f"  QNI Robustness Acc   : {qni_acc:.4f}")
    print(f"  Robustness drop      : {robust_drop:.2f} pp")

    lm = compute_latent_metrics(features, pred_probs)

    print(f"\n  Running t-SNE...")
    seed_all(42)   
    tsne = TSNE(
        n_components = 2,    
        perplexity   = 40,   
        n_iter       = 1000, 
        random_state = 42,   
        learning_rate= 200,  
        init         = 'pca' 
    )
    tsne_emb = tsne.fit_transform(features)   
    print(f"  t-SNE complete: {tsne_emb.shape}")

    return {
        'label'       : label,
        'features'    : features,       
        'tsne_emb'    : tsne_emb,       
        'true_labels' : true_labels,    
        'pred_labels' : pred_labels,    
        'pred_probs'  : pred_probs,     
        'clean_acc'   : overall_acc,
        'qni_acc'     : qni_acc,
        'robust_drop' : robust_drop,
        'report_dict' : report_dict,
        'latent'      : lm,
    }


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — INDIVIDUAL PLOTS 
# ══════════════════════════════════════════════════════════════════════════════

PALETTE = plt.cm.get_cmap('tab10', num_classes)   

def _style_ax(ax):
    ax.set_facecolor('#0f0f1a')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333355')
    ax.tick_params(colors='#888899')
    ax.yaxis.grid(True, linestyle=':', alpha=0.25, color='#555577')
    ax.set_axisbelow(True)

def plot_tsne_single(result, save_path):
    emb    = result['tsne_emb']       
    labels = result['true_labels']    
    label  = result['label']

    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#0f0f1a')

    for c in range(num_classes):
        mask = (labels == c)
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=[PALETTE(c)], label=class_names[c],
                   s=16, alpha=0.82, edgecolors='none')

    ax.set_title(f"t-SNE of GAP Features\n{label}",
                 fontsize=13, fontweight='bold', color='white', pad=12)
    ax.set_xlabel("t-SNE Dim 1", fontsize=10, color='#aaaacc')
    ax.set_ylabel("t-SNE Dim 2", fontsize=10, color='#aaaacc')
    _style_ax(ax)
    leg = ax.legend(title="Malware Family", fontsize=8, title_fontsize=9,
                    facecolor='#1a1a2e', edgecolor='#444466',
                    framealpha=0.4, labelcolor='white', loc='upper right')
    leg.get_title().set_color('white')
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_confusion_single(result, save_path):
    cm      = confusion_matrix(result['true_labels'], result['pred_labels'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) 

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#0f0f1a')

    sns.heatmap(cm_norm, annot=True, fmt='.2f',
                cmap='YlOrRd',                    
                vmin=0.0, vmax=1.0,               
                xticklabels=class_names,
                yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor='#1a1a2e',
                cbar_kws={'shrink': 0.8})

    ax.set_title(f"Confusion Matrix (row-normalised)\n{result['label']}",
                 fontsize=12, fontweight='bold', color='white', pad=12)
    ax.set_xlabel("Predicted", fontsize=10, color='#aaaacc', labelpad=8)
    ax.set_ylabel("True Label", fontsize=10, color='#aaaacc', labelpad=8)
    ax.tick_params(axis='x', colors='#ccccee', labelsize=8, rotation=40)
    ax.tick_params(axis='y', colors='#ccccee', labelsize=8, rotation=0)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=8, colors='#aaaacc')
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_f1_single(result, save_path):
    rd        = result['report_dict']
    f1_scores = [rd[cls]['f1-score'] for cls in class_names]
    macro_f1  = rd['macro avg']['f1-score']

    fig, ax = plt.subplots(figsize=(11, 4))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#0f0f1a')

    bar_colors = ['#4ade80' if f >= macro_f1 else '#fb923c' for f in f1_scores]
    bars = ax.bar(range(num_classes), f1_scores, color=bar_colors,
                  edgecolor='#222244', width=0.65)
    for bar, val in zip(bars, f1_scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom',
                fontsize=8, color='white', fontweight='bold')

    ax.axhline(y=macro_f1, color='#60a5fa', linestyle='--', linewidth=1.5,
               label=f'Macro F1 = {macro_f1:.3f}')
    ax.set_xticks(range(num_classes))
    ax.set_xticklabels(class_names, rotation=35, ha='right',
                       fontsize=8, color='#ccccee')
    ax.set_ylim(0, 1.08)   
    ax.set_ylabel("F1-Score", fontsize=10, color='#aaaacc')
    ax.set_title(f"Per-Class F1-Score\n{result['label']}",
                 fontsize=12, fontweight='bold', color='white', pad=10)
    ax.legend(fontsize=9, labelcolor='white', facecolor='#1a1a2e',
              edgecolor='#444466', framealpha=0.4)
    ax.tick_params(axis='y', colors='#aaaacc')
    _style_ax(ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — SIDE-BY-SIDE COMPARISON PLOTS
# ══════════════════════════════════════════════════════════════════════════════

def plot_tsne_comparison(r_clean, r_qni, save_path="comparison_tsne.png"):
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle(
        "t-SNE of GAP Features: Clean Baseline vs QNI-CCP Trained\n"
        "(Same random_state=42 — layout differences reflect learned feature structure)",
        fontsize=13, fontweight='bold', color='white', y=1.01
    )

    for ax, result in zip(axes, [r_clean, r_qni]):
        ax.set_facecolor('#0f0f1a')
        emb    = result['tsne_emb']
        labels = result['true_labels']

        for c in range(num_classes):
            mask = (labels == c)
            ax.scatter(emb[mask, 0], emb[mask, 1],
                       c=[PALETTE(c)], label=class_names[c],
                       s=16, alpha=0.82, edgecolors='none')

        stats = (f"Clean Acc: {result['clean_acc']:.4f}\n"
                 f"QNI Rob:   {result['qni_acc']:.4f}\n"
                 f"Δ drop:    {result['robust_drop']:.2f} pp")
        ax.text(0.02, 0.98, stats, transform=ax.transAxes,
                fontsize=9, color='#ccddff', va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#1a1a2e',
                          edgecolor='#444466', alpha=0.7))

        ax.set_title(result['label'], fontsize=12, fontweight='bold',
                     color='white', pad=10)
        ax.set_xlabel("t-SNE Dim 1", fontsize=10, color='#aaaacc')
        ax.set_ylabel("t-SNE Dim 2", fontsize=10, color='#aaaacc')
        _style_ax(ax)

    handles = [plt.scatter([], [], c=[PALETTE(c)], s=30, label=class_names[c])
               for c in range(num_classes)]
    fig.legend(handles=handles, title="Malware Family", fontsize=9,
               title_fontsize=10, ncol=5, loc='lower center',
               bbox_to_anchor=(0.5, -0.06),
               facecolor='#1a1a2e', edgecolor='#444466',
               framealpha=0.5, labelcolor='white').get_title().set_color('white')

    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_confusion_comparison(r_clean, r_qni,
                               save_path="comparison_confusion_matrix.png"):
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle("Confusion Matrix Comparison (row-normalised)",
                 fontsize=13, fontweight='bold', color='white')

    for ax, result in zip(axes, [r_clean, r_qni]):
        cm      = confusion_matrix(result['true_labels'], result['pred_labels'])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        ax.set_facecolor('#0f0f1a')

        sns.heatmap(cm_norm, annot=True, fmt='.2f',
                    cmap='YlOrRd', vmin=0.0, vmax=1.0,   
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.4, linecolor='#1a1a2e',
                    cbar_kws={'shrink': 0.8})

        ax.set_title(result['label'], fontsize=12, fontweight='bold',
                     color='white', pad=10)
        ax.set_xlabel("Predicted", fontsize=10, color='#aaaacc', labelpad=6)
        ax.set_ylabel("True Label", fontsize=10, color='#aaaacc', labelpad=6)
        ax.tick_params(axis='x', colors='#ccccee', labelsize=8, rotation=40)
        ax.tick_params(axis='y', colors='#ccccee', labelsize=8, rotation=0)
        ax.collections[0].colorbar.ax.tick_params(labelsize=8, colors='#aaaacc')

    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_f1_comparison(r_clean, r_qni,
                        save_path="comparison_per_class_f1.png"):
    rd_c = r_clean['report_dict']
    rd_q = r_qni['report_dict']

    f1_clean = np.array([rd_c[cls]['f1-score'] for cls in class_names])
    f1_qni   = np.array([rd_q[cls]['f1-score'] for cls in class_names])
    x        = np.arange(num_classes)
    w        = 0.38   

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#0f0f1a')

    bars_c = ax.bar(x - w/2, f1_clean, w, label=CLEAN_LABEL,
                    color='#f87171', edgecolor='#222244', alpha=0.9)
    bars_q = ax.bar(x + w/2, f1_qni,   w, label=QNI_LABEL,
                    color='#4ade80', edgecolor='#222244', alpha=0.9)

    for bar in bars_c:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom',
                fontsize=7, color='#ffaaaa')
    for bar in bars_q:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom',
                fontsize=7, color='#aaffaa')

    ax.axhline(rd_c['macro avg']['f1-score'], color='#f87171',
               linestyle='--', linewidth=1.2, alpha=0.6,
               label=f'Clean macro avg = {rd_c["macro avg"]["f1-score"]:.3f}')
    ax.axhline(rd_q['macro avg']['f1-score'], color='#4ade80',
               linestyle='--', linewidth=1.2, alpha=0.6,
               label=f'QNI macro avg = {rd_q["macro avg"]["f1-score"]:.3f}')

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=35, ha='right',
                       fontsize=8, color='#ccccee')
    ax.set_ylim(0, 1.10)
    ax.set_ylabel("F1-Score", fontsize=10, color='#aaaacc')
    ax.set_title("Per-Class F1-Score: Clean Baseline vs QNI-CCP Trained",
                 fontsize=12, fontweight='bold', color='white', pad=10)
    ax.legend(fontsize=9, labelcolor='white', facecolor='#1a1a2e',
              edgecolor='#444466', framealpha=0.5, ncol=2)
    ax.tick_params(axis='y', colors='#aaaacc')
    _style_ax(ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")

def plot_latent_metrics_comparison(r_clean, r_qni,
                                    save_path="comparison_latent_metrics.png"):
    lm_c = r_clean['latent']
    lm_q = r_qni['latent']

    metrics_scalar = [
        ('entropy',          'Entropy',               False), 
        ('feature_variance', 'Feature Variance',        False), 
        ('avg_confidence',   'Avg Confidence',          True),  
        ('conf_std',         'Confidence Std Dev',      False), 
    ]
    metrics_pct = [
        ('high_conf_pct',    'High-Conf Patterns (%)', True),  
    ]

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle(
        "Latent-Space Metrics Comparison (Table 6 style)\n"
        f"{CLEAN_LABEL}  vs  {QNI_LABEL}",
        fontsize=12, fontweight='bold', color='white'
    )

    all_metrics = metrics_scalar + metrics_pct
    for i, (key, title, higher_is_better) in enumerate(all_metrics):
        ax  = axes[i]
        ax.set_facecolor('#0f0f1a')

        val_c = lm_c[key]
        val_q = lm_q[key]

        if higher_is_better:
            col_c = '#4ade80' if val_c >= val_q else '#f87171'
            col_q = '#4ade80' if val_q >= val_c else '#f87171'
        else:
            col_c = '#4ade80' if val_c <= val_q else '#f87171'
            col_q = '#4ade80' if val_q <= val_c else '#f87171'

        bars = ax.bar([CLEAN_LABEL, QNI_LABEL], [val_c, val_q],
                      color=[col_c, col_q], edgecolor='#222244', width=0.5)
        for bar, val in zip(bars, [val_c, val_q]):
            label = f'{val:.1f}%' if key == 'high_conf_pct' else f'{val:.4f}'
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(val_c, val_q) * 0.03,
                    label, ha='center', va='bottom',
                    fontsize=9, color='white', fontweight='bold')

        ax.set_title(title, fontsize=10, fontweight='bold',
                     color='white', pad=8)
        ax.set_ylim(0, max(val_c, val_q) * 1.25 + 1e-6)
        ax.tick_params(axis='x', colors='#ccccee', labelsize=8, rotation=15)
        ax.tick_params(axis='y', colors='#888899', labelsize=8)
        _style_ax(ax)

    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY TABLE 
# ══════════════════════════════════════════════════════════════════════════════

def print_comparison_table(r_clean, r_qni):
    rc = r_clean
    rq = r_qni

    def delta(a, b, higher_better=True):
        d = b - a
        arrow = '↑' if (d > 0) == higher_better else '↓'
        return f"{d:+.4f} {arrow}"

    print("\n" + "═"*72)
    print(f"  COMPARISON SUMMARY: {CLEAN_LABEL}  →  {QNI_LABEL}")
    print("═"*72)
    print(f"  {'Metric':<35} {'Clean':>10} {'QNI-CCP':>10} {'Δ':>14}")
    print(f"  {'─'*65}")

    print(f"  {'Clean Test Accuracy':<35} "
          f"{rc['clean_acc']:>10.4f} {rq['clean_acc']:>10.4f} "
          f"{delta(rc['clean_acc'], rq['clean_acc']):>14}")
    print(f"  {'QNI Robustness Accuracy':<35} "
          f"{rc['qni_acc']:>10.4f} {rq['qni_acc']:>10.4f} "
          f"{delta(rc['qni_acc'], rq['qni_acc']):>14}")
    print(f"  {'Robustness Drop (pp)':<35} "
          f"{rc['robust_drop']:>10.2f} {rq['robust_drop']:>10.2f} "
          f"{delta(rc['robust_drop'], rq['robust_drop'], higher_better=False):>14}")

    print(f"  {'─'*65}")
    for key, name, hb in [
        ('macro avg',    'Macro Precision',  True),
        ('macro avg',    'Macro Recall',     True),
        ('macro avg',    'Macro F1',         True),
        ('weighted avg', 'Weighted F1',      True),
    ]:
        sub = 'precision' if 'Precision' in name else \
              'recall'    if 'Recall'    in name else 'f1-score'
        vc = rc['report_dict'][key][sub]
        vq = rq['report_dict'][key][sub]
        print(f"  {name:<35} {vc:>10.4f} {vq:>10.4f} {delta(vc, vq, hb):>14}")

    print(f"  {'─'*65}")
    for key, name, hb in [
        ('entropy',          'Entropy',              False),
        ('feature_variance', 'Feature Variance',     False),
        ('unique_patterns',  'Unique Predicted Cls', None), # Updated Name
        ('high_conf_pct',    'High-Conf (Median) %', True), # Updated Name
        ('avg_confidence',   'Avg Confidence',       True),
        ('conf_std',         'Confidence Std Dev',   False),
    ]:
        vc = rc['latent'][key]
        vq = rq['latent'][key]
        if key == 'high_conf_pct':
            print(f"  {name:<35} {vc:>9.1f}% {vq:>9.1f}% "
                  f"{delta(vc, vq, hb):>14}")
        elif key == 'unique_patterns':
            print(f"  {name:<35} {int(vc):>10d} {int(vq):>10d}            —")
        else:
            print(f"  {name:<35} {vc:>10.4f} {vq:>10.4f} "
                  f"{delta(vc, vq, hb) if hb is not None else '—':>14}")

    print("═"*72)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def load_model(path, label):
    m = HybridResNet(n_qubits, q_out_dim, num_classes, dropout=0.35).to(device)
    ckpt = torch.load(path, map_location=device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    print(f"  Loaded [{label}] from {path}  "
          f"(epoch={ckpt.get('epoch','?')}, val_acc={ckpt.get('val_acc','?')})")
    return m

print("\n" + "═"*60)
print("  LOADING MODELS")
print("═"*60)
model_clean = load_model(CLEAN_CHECKPOINT, CLEAN_LABEL)
model_qni   = load_model(QNI_CHECKPOINT,   QNI_LABEL)

print("\n" + "═"*60)
print("  EVALUATING CLEAN BASELINE")
print("═"*60)
r_clean = evaluate_model(model_clean, CLEAN_LABEL, test_loader, train_loader, device)

print("\n" + "═"*60)
print("  EVALUATING QNI-CCP MODEL")
print("═"*60)
r_qni = evaluate_model(model_qni, QNI_LABEL, test_loader, train_loader, device)

print_comparison_table(r_clean, r_qni)

print("\n" + "═"*60)
print("  SAVING INDIVIDUAL PLOTS")
print("═"*60)
plot_tsne_single(r_clean, "clean_tsne.png")
plot_tsne_single(r_qni,   "qni_tsne.png")
plot_confusion_single(r_clean, "clean_confusion_matrix.png")
plot_confusion_single(r_qni,   "qni_confusion_matrix.png")
plot_f1_single(r_clean, "clean_per_class_f1.png")
plot_f1_single(r_qni,   "qni_per_class_f1.png")

print("\n" + "═"*60)
print("  SAVING COMPARISON PLOTS")
print("═"*60)
plot_tsne_comparison(r_clean, r_qni,        "comparison_tsne.png")
plot_confusion_comparison(r_clean, r_qni,   "comparison_confusion_matrix.png")
plot_f1_comparison(r_clean, r_qni,          "comparison_per_class_f1.png")
plot_latent_metrics_comparison(r_clean, r_qni, "comparison_latent_metrics.png")

print("\n✅ All evaluations complete.")

Device: cuda
Test  samples : 3458
Train samples : 48498  (used only for centroids)
Classes (10): ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

════════════════════════════════════════════════════════════
  LOADING MODELS
════════════════════════════════════════════════════════════
  Loaded [Clean Baseline] from hybrid_resnet_NOGAN.pth  (epoch=55, val_acc=0.9177175935497209)
  Loaded [QNI-CCP Trained] from QNI1.pth  (epoch=15, val_acc=0.9241265247053959)

════════════════════════════════════════════════════════════
  EVALUATING CLEAN BASELINE
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Evaluating: Clean Baseline
────────────────────────────────────────────────────────────


  Features extracted: (3458, 64)

  Classification Report — Clean Baseline
              precision    recall  f1-score   support

           0     0.5000    0.4171    0.4548       175
           1     0.9980    0.9960    0.9970       496
           2     0.8500    0.9951    0.9169       205
           3     0.9497    0.9659    0.9577       176
           4     1.0000    1.0000    1.0000        58
           5     0.9265    0.9123    0.9193       456
           6     0.9649    0.9332    0.9488      1003
           7     0.8987    0.8676    0.8829       491
           8     0.8602    0.9249    0.8914       173
           9     0.8199    0.9511    0.8807       225

    accuracy                         0.9112      3458
   macro avg     0.8768    0.8963    0.8849      3458
weighted avg     0.9100    0.9112    0.9096      3458

  Computing centroids for QNI robustness...


  Clean Accuracy       : 0.9112
  QNI Robustness Acc   : 0.9057
  Robustness drop      : 0.55 pp

  Running t-SNE...
  t-SNE complete: (3458, 2)

════════════════════════════════════════════════════════════
  EVALUATING QNI-CCP MODEL
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Evaluating: QNI-CCP Trained
────────────────────────────────────────────────────────────


  Features extracted: (3458, 64)

  Classification Report — QNI-CCP Trained
              precision    recall  f1-score   support

           0     0.5079    0.3657    0.4252       175
           1     1.0000    0.9940    0.9970       496
           2     0.9018    0.9854    0.9417       205
           3     0.9659    0.9659    0.9659       176
           4     0.9831    1.0000    0.9915        58
           5     0.9587    0.9167    0.9372       456
           6     0.9552    0.9352    0.9451      1003
           7     0.8878    0.9185    0.9029       491
           8     0.8256    0.9306    0.8750       173
           9     0.8263    0.9511    0.8843       225

    accuracy                         0.9164      3458
   macro avg     0.8812    0.8963    0.8866      3458
weighted avg     0.9129    0.9164    0.9134      3458

  Computing centroids for QNI robustness...


  Clean Accuracy       : 0.9164
  QNI Robustness Acc   : 0.8997
  Robustness drop      : 1.68 pp

  Running t-SNE...
  t-SNE complete: (3458, 2)

════════════════════════════════════════════════════════════════════════
  COMPARISON SUMMARY: Clean Baseline  →  QNI-CCP Trained
════════════════════════════════════════════════════════════════════════
  Metric                                   Clean    QNI-CCP              Δ
  ─────────────────────────────────────────────────────────────────
  Clean Test Accuracy                     0.9112     0.9164      +0.0052 ↑
  QNI Robustness Accuracy                 0.9057     0.8997      -0.0061 ↓
  Robustness Drop (pp)                      0.55       1.68      +1.1278 ↓
  ─────────────────────────────────────────────────────────────────
  Macro Precision                         0.8768     0.8812      +0.0044 ↑
  Macro Recall                            0.8963     0.8963      -0.0000 ↓
  Macro F1                                0.8849     0.8866      